In [1]:
!podman ps

CONTAINER ID  IMAGE                             COMMAND               CREATED      STATUS                   PORTS                                                                   NAMES
0569012855d8  docker.io/library/adminer:latest  php -S [::]:8080 ...  5 weeks ago  Up 13 seconds            0.0.0.0:8080->8080/tcp                                                  adminer-ui
61783c60dfcb  docker.io/ipfs/kubo:latest        daemon --migrate=...  5 weeks ago  Up 13 seconds (healthy)  0.0.0.0:4001->4001/tcp, 0.0.0.0:5001->5001/tcp, 0.0.0.0:8081->8080/tcp  cobalt_protocol_kubo_dev
cf0033527254  docker.io/pgvector/pgvector:pg17  postgres              7 days ago   Up 13 seconds            0.0.0.0:5432->5432/tcp                                                  pgvector-1


In [2]:
!python --version

Python 3.12.14


In [3]:
%pip install pymupdf openai python-dotenv requests

Note: you may need to restart the kernel to use updated packages.


In [23]:
import os
import re
import base64
import pymupdf
import requests
import io
import ast
import json

from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

In [7]:
load_dotenv()

client = OpenAI(
    api_key=os.environ["CHUTES_API_KEY"],
    base_url="https://llm.chutes.ai/v1",
)

MODEL = "Qwen/Qwen3.5-397B-A17B-TEE"

FORMAT_GAMBAR = {
    ".png": "image/png",
    ".jpg": "image/jpeg",
    ".jpeg": "image/jpeg",
    ".webp": "image/webp",
}

In [8]:
def bersihkan_output(teks):
    teks = "\n".join(
        baris.strip()
        for baris in teks.splitlines()
    )

    teks = re.sub(r"\n{3,}", "\n\n", teks)

    return teks.strip()

In [9]:
def baca_dokumen(path, max_pages=None):

    # 1. Buka dokumen berdasarkan tipe input
    if isinstance(path, io.BytesIO):
        dokumen = pymupdf.open(
            stream=path.getvalue(),
            filetype="pdf"
        )

    elif isinstance(path, (bytes, bytearray)):
        dokumen = pymupdf.open(
            stream=path,
            filetype="pdf"
        )

    else:
        path = Path(path)

        if not path.exists():
            raise FileNotFoundError(
                f"File tidak ditemukan: {path}"
            )

        ekstensi = path.suffix.lower()

        format_gambar = {
            ".png",
            ".jpg",
            ".jpeg",
            ".webp"
        }

        if ekstensi != ".pdf" and ekstensi not in format_gambar:
            raise ValueError(
                "Format harus PDF, PNG, JPG, JPEG, atau WEBP."
            )

        dokumen = pymupdf.open(str(path))

    try:
        jumlah_halaman = len(dokumen)

        if max_pages is not None:
            jumlah_halaman = min(
                jumlah_halaman,
                max_pages
            )

        konteks_teks = []
        konten_visual = []

        for i in range(jumlah_halaman):

            halaman = dokumen[i]

            # 1. Extract text
            teks = halaman.get_text("text").strip()

            if teks:
                konteks_teks.append(
                    f"Halaman {i + 1}:\n{teks}"
                )

            # 2. Render halaman menjadi gambar
            pix = halaman.get_pixmap(
                matrix=pymupdf.Matrix(1.5, 1.5),
                alpha=False
            )

            gambar = base64.b64encode(
                pix.tobytes("png")
            ).decode("utf-8")

            konten_visual.append({
                "type": "text",
                "text": f"Visual halaman {i + 1}:"
            })

            konten_visual.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{gambar}"
                }
            })

        teks_pdf = "\n\n".join(konteks_teks)

        return teks_pdf, konten_visual

    finally:
        dokumen.close()

In [19]:
def analisis_dokumen(path, pertanyaan, max_pages=None):

    # 1. Buka dokumen berdasarkan jenis input
    if isinstance(path, io.BytesIO):

        dokumen = pymupdf.open(
            stream=path.getvalue(),
            filetype="pdf"
        )

    elif isinstance(path, (bytes, bytearray)):

        dokumen = pymupdf.open(
            stream=path,
            filetype="pdf"
        )

    else:

        path = Path(path)
        ekstensi = path.suffix.lower()

        if not path.exists():
            raise FileNotFoundError(
                f"File tidak ditemukan: {path}"
            )

        if ekstensi != ".pdf" and ekstensi not in FORMAT_GAMBAR:
            raise ValueError(
                "Format harus PDF, PNG, JPG, JPEG, atau WEBP."
            )

        dokumen = pymupdf.open(str(path))

    try:

        # 2. Batasi jumlah halaman
        jumlah_halaman = len(dokumen)

        if max_pages is not None:
            jumlah_halaman = min(
                jumlah_halaman,
                max_pages
            )

        konteks_teks = []
        konten_visual = []

        # 3. Extract teks dan render gambar
        for i in range(jumlah_halaman):

            halaman = dokumen[i]

            # Extract text
            teks = halaman.get_text("text").strip()

            if teks:
                konteks_teks.append(
                    f"Halaman {i + 1}:\n{teks}"
                )

            # Render halaman menjadi gambar
            pix = halaman.get_pixmap(
                matrix=pymupdf.Matrix(1.5, 1.5),
                alpha=False
            )

            gambar = base64.b64encode(
                pix.tobytes("png")
            ).decode("utf-8")

            konten_visual.append({
                "type": "text",
                "text": f"Visual halaman {i + 1}:"
            })

            konten_visual.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{gambar}"
                }
            })

        teks_pdf = "\n\n".join(konteks_teks)

        if not teks_pdf:
            teks_pdf = (
                "Tidak ada teks yang berhasil diekstrak. "
                "Gunakan gambar untuk membaca dokumen."
            )

    finally:
        dokumen.close()

    # 5. Gabungkan pertanyaan, teks, dan gambar
    konten_user = [
        {
            "type": "text",
            "text": (
                "Kamu bertugas memahami dan menjawab pertanyaan "
                "berdasarkan dokumen yang diberikan.\n\n"

                f"PERTANYAAN:\n{pertanyaan}\n\n"

                f"ISI DOKUMEN:\n{teks_pdf}\n\n"

                "ATURAN PENILAIAN:\n"
                "1. relevan = y jika topik atau maksud pertanyaan "
                "berhubungan dengan isi, tujuan, objek, atau pembahasan dokumen.\n"

                "2. relevan = n jika pertanyaan sama sekali tidak "
                "berhubungan dengan dokumen.\n"

                "3. status = not_found jika pertanyaan relevan "
                "tetapi informasi yang diminta tidak ditemukan.\n\n"

                "ATURAN MENJAWAB:\n"
                "- Jika relevan = y, jawab pertanyaan berdasarkan dokumen.\n"
                "- Jika relevan = n, jawab bahwa pertanyaan tidak relevan "
                "dengan dokumen, kemudian berikan ringkasan singkat dokumen.\n"
                "- Jika status = not_found, jelaskan bahwa informasi "
                "tidak ditemukan dalam dokumen.\n"
                "- Jangan mengarang informasi.\n"
                "- Gunakan teks dan visual sebagai sumber informasi.\n\n"

                "FORMAT OUTPUT WAJIB JSON:\n"
                "{\n"
                '  "relevan": "y atau n",\n'
                '  "status": "answered atau not_found atau irrelevant",\n'
                '  "jawaban": "jawaban atau penjelasan"\n'
                "}\n\n"

                "Jangan gunakan Markdown code block. "
                "Jangan memberikan teks di luar JSON."
            )
        }
    ]

    konten_user.extend(konten_visual)

    # 6. Kirim ke Vision LLM
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Kamu adalah document understanding assistant. "
                    "Tugasmu mengidentifikasi hubungan pertanyaan dengan "
                    "dokumen, kemudian menjawab berdasarkan bukti tekstual "
                    "dan visual. Jangan menyimpulkan bahwa pertanyaan tidak "
                    "relevan hanya karena jawaban spesifiknya tidak ditemukan. "
                    "Selalu keluarkan JSON valid."
                )
            },
            {
                "role": "user",
                "content": konten_user
            }
        ],
        response_format={"type": "json_object"},
        max_tokens=1500
    )

    return bersihkan_output(
        response.choices[0].message.content or ""
    )

In [21]:
def download_pdf(url):
    response = requests.get(url, timeout=60)
    response.raise_for_status()

    pdf_memory = io.BytesIO(response.content)

    return pdf_memory

In [24]:
# Menggunakan PDF dari URL
pdf_memory = download_pdf(
    "http://localhost:8081/ipfs/QmQVXAukDkho9L84gNZTgXNviny3gqyegNDP32DXk5kicx"
)

jawaban = analisis_dokumen(
    path=pdf_memory,
    pertanyaan="apa isi dokumen tersebut?"
)

try:
    json_jawaban = ast.literal_eval(jawaban)
except SyntaxError:
    json_jawaban = json.loads(jawaban)

print(json_jawaban)

{'relevan': 'y', 'status': 'answered', 'jawaban': "Dokumen ini berisi Skema Okupasi untuk Associate Programmer dari LSP Programming Universitas Gunadarma. Isinya adalah Skenario Tugas Terstruktur yang meminta peserta (sebagai Programmer Muda) untuk membuat sistem terkomputerisasi untuk Restoran 'Dapur Ina Aina'. Dokumen ini merinci 4 tugas utama: 1) Memilih metode pengembangan aplikasi dan membuat diagram rancangan sistem. 2) Menentukan tipe data berbasis kelas (OOP) dan merancang database. 3) Mengimplementasikan program untuk transaksi pesanan, update stok makanan/minuman, dan menampilkan billing. 4) Melakukan pengujian unit program dan membuat dokumentasi hasil uji."}


In [25]:
# Menggunakan PDF dari local
jawaban = analisis_dokumen(
    path="2. Tugas Studi Kasus  Skema programmer-Dapur Ina Aina.pdf",
    pertanyaan="Apa isi dokumen ini?"
)

try:
    json_jawaban = ast.literal_eval(jawaban)
except SyntaxError:
    json_jawaban = json.loads(jawaban)

print(json_jawaban)

{'relevan': 'y', 'status': 'answered', 'jawaban': "Dokumen ini berjudul 'SKEMA OKUPASI ASSOCIATE PROGRAMMER' dari LSP Programming Universitas Gunadarma. Isi dokumen tersebut merupakan 'Skenario Tugas Terstruktur' yang berisi studi kasus dan instruksi tugas untuk seorang Programmer Muda. \n\nRincian isi dokumen adalah sebagai berikut:\n1. **Skenario**: Restoran 'Dapur Ina Aina' membutuhkan sistem terkomputerisasi untuk operasional (pemesanan, transaksi, stok, laporan). Pembayaran bisa tunai atau non-tunai.\n2. **Tugas 1**: Memilih metode pengembangan aplikasi, membuat diagram pemodelan objek, dan merancang sistem.\n3. **Tugas 2**: Menentukan tipe data berdasarkan class (OOP) dan merancang database (tabel, field, tipe, length).\n4. **Tugas 3**: Implementasi program yang mencakup:\n   - Program pengolahan transaksi (data pesanan pelanggan).\n   - Program input dan update data stok makanan/minuman.\n   - Program menampilkan billing pesanan dengan opsi pembayaran tunai/non-tunai.\n5. **Tuga

In [ ]:
# Menggunakan Gambar dari local
jawaban = analisis_dokumen(
    path="1f9e63d3-20e8-42cb-b676-fd985fdcbccc.jpeg",
    pertanyaan="siapa presiden indonesia pertama?"
)

try:
    json_jawaban = ast.literal_eval(jawaban)
except SyntaxError:
    json_jawaban = json.loads(jawaban)

print(json_jawaban)

{'relevan': 'n', 'status': 'irrelevant', 'jawaban': 'Pertanyaan tidak relevan dengan dokumen. Dokumen ini adalah poster promosi layanan top up Valorant Point melalui Rakit App yang menampilkan berbagai pilihan paket poin dan harganya dalam Rupiah.'}


In [29]:
pdf_memory = download_pdf(
    "http://localhost:8081/ipfs/QmQVXAukDkho9L84gNZTgXNviny3gqyegNDP32DXk5kicx"
)

teks_pdf, konten_visual = baca_dokumen(pdf_memory)

print(teks_pdf)

Halaman 1:
SKEMA OKUPASI 
ASSOCIATE PROGRAMMER

Halaman 2:
LSP |  Programming  |  Universitas Gunadarma
• Restoran “Dapur Ina Aina” hendak membuat sebuah system terkomputerisasi berhubungan dengan kegiatan
operasional dan transaksi seperti Pemesanan oleh pelanggan, Transaksi penjualan, Informasi stok barang, dan
Laporan penjulan secara berkala. Proses transaksi dimulai dari pesanan yang dilakukan oleh pelanggan sampai
dihasilkannya billing atau tagihan kepada pelanggan. Pembayaran yang dlakukan dapat menggunakan Tunai dan
Non Tunai (Debit/Kartu Kredit). Administrator mengelola stok makanan dan minuman yang tersedia dan habis.
Laporan penjualan dilakukan secara berkala perminggu atau perbulan. Adapaun kategori produk yang dijual
didefinisikan dalam 3 (tiga) kategori yaitu Makanan utama, Appetizer dan Minuman.
• Sebagai seorang Programmer Muda , Anda adalah orang yang bertanggung jawab terhadap implementasi dari
system terkomputerisasi tersebut, silahkan lakukan dan definisikan spesifika